# Canadian Multi-City Housing Listings — Portfolio Workflow

**Signal to hiring managers:** This is a **repeatable analytics workflow**: ingestion → quality checks → EDA → unsupervised segmentation → **interpretable regression** → **validated ML** (group-aware splits).

### What problem does this solve?
We quantify how **walkability**, **dwelling attributes**, and **metro** relate to **asking price**, and we build models whose performance is measured **honestly** (held-out cities in expectation via grouped split).

### Deliverables in this notebook
| Phase | Output |
|-------|--------|
| 1–2 | Config + embedded `prepare_listings()` pipeline |
| 3 | Data-quality dashboard |
| 4 | Core visuals (distribution + relationships) |
| 5 | Market snapshot metrics |
| 6 | K-Means listing personas |
| 7 | OLS on log-price — walk score “premium” |
| 8 | Ridge / Random Forest / Gradient Boosting — **GroupShuffleSplit by `city`** |
| 9 | Feature importance (Random Forest, per city) |
| 10 | Interview-ready recap |

## Phase 1 — Setup

**Requirements:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`, `statsmodels`

The next cell contains the **full data-prep module** (same source as `listing_pipeline.py` in this folder) so you can submit **one notebook** or keep the `.py` file for import elsewhere.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler as SKStandardScaler
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")

DATA_CANDIDATES = [
    Path("housing_clean_4b.csv"),
    Path("final_data_refactored.csv"),
    Path("listings_prepared.csv"),
]

In [ ]:
"""
Listing data preparation — shared by Housing_Portfolio_Workflow.ipynb.
Consolidates parsing, imputation, and outlier rules from the course scraper pipeline.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


def prepare_listings(raw: pd.DataFrame) -> pd.DataFrame:
    """Clean Zolo-style listings into analysis-ready columns (adds *_num aliases)."""
    df = raw.copy()

    if "neighbouhood" in df.columns and "neighbourhood" not in df.columns:
        df = df.rename(columns={"neighbouhood": "neighbourhood"})

    obj = df.select_dtypes(include=["object", "string"]).columns
    if len(obj):
        df[obj] = df[obj].apply(lambda s: s.astype("string").str.strip())
        tokens = {"", "na", "n/a", "none", "no data", "unknown", "xxxxxx"}
        mask = df[obj].apply(lambda s: s.str.strip().str.lower().isin(tokens))
        df[obj] = df[obj].mask(mask, np.nan)

    for c in ["address", "city", "neighbourhood", "type", "style"]:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.title()

    if "bed" in df.columns:
        m = df["bed"].astype(str).str.extract(r"(\d+)\s*\+\s*(\d+)")
        s = df["bed"].astype(str).str.extract(r"(\d+)")[0]
        df["bed_main"] = pd.to_numeric(m[0], errors="coerce").fillna(pd.to_numeric(s, errors="coerce"))
        df["bed_plus"] = pd.to_numeric(m[1], errors="coerce").fillna(0)
        df["bed_total"] = df["bed_main"].fillna(0) + df["bed_plus"]

    if "size" in df.columns:
        s = (
            df["size"]
            .astype("string")
            .str.lower()
            .str.replace(",", "", regex=False)
            .str.replace(r"\s+", " ", regex=True)
        )
        r = s.str.extract(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)")
        lo, hi = pd.to_numeric(r[0], errors="coerce"), pd.to_numeric(r[1], errors="coerce")
        lt = pd.to_numeric(s.str.extract(r"<\s*(\d+(?:\.\d+)?)")[0], errors="coerce")
        pl = pd.to_numeric(s.str.extract(r"(\d+(?:\.\d+)?)\s*\+")[0], errors="coerce")
        single = pd.to_numeric(s.str.extract(r"^\s*(\d+(?:\.\d+)?)\s*$")[0], errors="coerce")
        df["size_min"] = lo.combine_first(pl)
        df["size_max"] = hi.combine_first(lt)
        df["size_single"] = single
        df["size_mid"] = (
            df[["size_min", "size_max"]]
            .mean(axis=1)
            .fillna(df["size_min"])
            .fillna(df["size_max"])
            .fillna(df["size_single"])
        )

    if "walk_score" in df.columns:
        df["walk_score"] = pd.to_numeric(
            df["walk_score"].replace({"—": np.nan, "–": np.nan}), errors="coerce"
        )
        df.loc[(df["walk_score"] < 0) | (df["walk_score"] > 100), "walk_score"] = np.nan

    if "lot_size" in df.columns:
        s = (
            df["lot_size"]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace("×", "x")
            .str.replace(r"\s+", " ", regex=True)
        )
        sq = pd.to_numeric(
            s.str.extract(r"(\d+(?:\.\d+)?)\s*(?:sq\.?\s*ft|sqft|sf|ft2|ft²)\b")[0],
            errors="coerce",
        )
        ac = pd.to_numeric(s.str.extract(r"(\d+(?:\.\d+)?)\s*a(?:cre)?s?\b")[0], errors="coerce")
        sqm = pd.to_numeric(s.str.extract(r"(\d+(?:\.\d+)?)\s*(?:m2|m²|sqm)\b")[0], errors="coerce")
        ha = pd.to_numeric(s.str.extract(r"(\d+(?:\.\d+)?)\s*ha\b")[0], errors="coerce")
        dm = s.str.extract(r"(\d+(?:\.\d+)?)\s*[x*]\s*(\d+(?:\.\d+)?)")
        a = pd.to_numeric(dm[0], errors="coerce")
        b = pd.to_numeric(dm[1], errors="coerce")
        dims = (a * b).where(
            ~s.str.contains(r"\b(?:m2|m²|sqm)\b", na=False), (a * b) * 10.7639
        )
        dims = dims.mask((a <= 0) | (b <= 0) | a.isna() | b.isna())
        la = sq.combine_first(ac * 43560).combine_first(sqm * 10.7639).combine_first(ha * 107639).combine_first(dims)
        amb = a.notna() & b.notna() & s.str.contains(r"\b(?:acres?|ha)\b", na=False)
        df["lot_area_sqft"] = la.mask(amb | (la <= 1) | (la > 2_000_000)).astype("Float64")

    if "age" in df.columns:
        age = df["age"].astype("string").str.lower().str.replace(r"\s+", " ", regex=True)
        r = age.str.extract(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)")
        lo, hi = pd.to_numeric(r[0], errors="coerce"), pd.to_numeric(r[1], errors="coerce")
        p = pd.to_numeric(age.str.extract(r"(\d+(?:\.\d+)?)\s*\+")[0], errors="coerce")
        sgl = pd.to_numeric(
            age.str.extract(r"(\d+(?:\.\d+)?)(?:\s*(?:years?|yrs?)\b)?$")[0], errors="coerce"
        )
        tmp = lo.combine_first(p).combine_first(sgl).where(hi.isna(), (lo + hi) / 2.0)
        df["age_years"] = tmp.where((tmp >= 0) & (tmp <= 150)).astype("Float64")

    if "listed_price" in df.columns:
        df["listed_price"] = pd.to_numeric(
            df["listed_price"].astype(str).str.replace(r"[^\d.]", "", regex=True),
            errors="coerce",
        )

    if "property_tax" in df.columns:
        df["property_tax"] = pd.to_numeric(
            df["property_tax"].astype(str).str.replace(r"[^\d.]", "", regex=True),
            errors="coerce",
        )

    if "bath" in df.columns:
        df["bath"] = pd.to_numeric(df["bath"], errors="coerce")

    protected = ["lot_area_sqft", "age_years"]
    num = [c for c in df.select_dtypes(include=[np.number]).columns if c not in protected]
    if num:
        df[num] = df[num].astype("Float64")
        if {"city", "type"}.issubset(df.columns):
            medg = df.groupby(["city", "type"])[num].transform(lambda s: s.dropna().median())
            df[num] = df[num].fillna(medg)
        df[num] = df[num].fillna(df[num].median(numeric_only=True))

    if "bed_total" in df.columns:
        m = (df["bed_total"] < 0) | (df["bed_total"] > 10)
        df.loc[m, "bed_total"] = np.nan
    if "size_mid" in df.columns:
        m = (df["size_mid"] < 100) | (df["size_mid"] > 20000)
        df.loc[m, "size_mid"] = np.nan
    if "walk_score" in df.columns:
        m = (df["walk_score"] < 0) | (df["walk_score"] > 100)
        df.loc[m, "walk_score"] = np.nan
    if "listed_price" in df.columns:
        m = (df["listed_price"] < 1e4) | (df["listed_price"] > 1e8)
        df.loc[m, "listed_price"] = np.nan
    if {"listed_price", "size_mid"}.issubset(df.columns):
        df["price_per_sqft"] = df["listed_price"] / df["size_mid"]
        m = (df["price_per_sqft"] < 10) | (df["price_per_sqft"] > 5000)
        df.loc[m, "price_per_sqft"] = np.nan
    if {"property_tax", "listed_price"}.issubset(df.columns):
        df.loc[df["property_tax"] == 1, "property_tax"] = np.nan
        m = df["listed_price"].notna() & (df["property_tax"] > 0.10 * df["listed_price"])
        df.loc[m, "property_tax"] = np.nan

    num2 = [c for c in df.select_dtypes(include=[np.number]).columns if c not in protected]
    if num2:
        df[num2] = df[num2].astype("Float64")
        if {"city", "type"}.issubset(df.columns):
            df[num2] = df.groupby(["city", "type"])[num2].transform(lambda s: s.fillna(s.median()))
        df[num2] = df[num2].fillna(df[num2].median(numeric_only=True))

    df["listed_price_num"] = df["listed_price"] if "listed_price" in df.columns else np.nan
    df["walk_score_num"] = df["walk_score"] if "walk_score" in df.columns else np.nan
    df["bath_num"] = df["bath"] if "bath" in df.columns else np.nan
    df["bed_num"] = df["bed_total"] if "bed_total" in df.columns else np.nan
    if "property_tax" in df.columns:
        df["property_tax_num"] = df["property_tax"]

    df["bath_num"] = df["bath_num"].clip(lower=0.5, upper=12)
    df["price_log"] = np.log(df["listed_price_num"].where(df["listed_price_num"] > 0))
    if "lot_area_sqft" in df.columns:
        df["lot_sqft"] = df["lot_area_sqft"]

    return df


def load_prepared_csv(path: str | Path) -> pd.DataFrame:
    return pd.read_csv(Path(path), low_memory=False)

## Phase 2 — Load and prepare

Set **`DATA_CANDIDATES`** to your export path(s). If you are coming from the midterm notebook, run its **“save CSV”** cell first (e.g. `final_data_refactored.csv`).

`prepare_listings()` harmonizes **raw or partially cleaned** exports into modeling columns (`*_num`, `price_per_sqft`, etc.).

In [ ]:
def resolve_dataframe(paths):
    for p in paths:
        if p.exists():
            print(f"Using: {p.resolve()}") 
            raw = pd.read_csv(p, low_memory=False)
            return prepare_listings(raw)
    raise FileNotFoundError(
        "No data file found. Place one of: "
        + ", ".join(str(p) for p in paths)
        + " (or update DATA_CANDIDATES)."
    )

df = resolve_dataframe(DATA_CANDIDATES)
print("Shape:", df.shape)

## Phase 3 — Data quality dashboard

In [ ]:
qc = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "missing_pct": (df.isna().mean() * 100).round(2),
})
qc = qc.sort_values("missing_pct", ascending=False)
display(qc.head(20))

key = [
    "listed_price_num", "walk_score_num", "size_mid",
    "bath_num", "bed_num", "city", "type", "price_per_sqft",
]
print("\nKey columns present:", [c for c in key if c in df.columns])

## Phase 4 — Core EDA

**Principle for portfolios:** a few **high-information** plots beat dozens of repetitive charts.

In [ ]:
eda = df.dropna(subset=["listed_price_num", "city"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(eda["listed_price_num"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Listed price (long right tail)")
axes[0].set_xlabel("Price (CAD)")

sns.boxplot(data=eda, x="city", y="listed_price_num", ax=axes[1])
axes[1].set_title("Price by metro")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

pair = df.dropna(subset=["size_mid", "listed_price_num", "walk_score_num"])
fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(
    pair["size_mid"], pair["listed_price_num"],
    c=pair["walk_score_num"], cmap="viridis", alpha=0.35, s=12
)
plt.colorbar(sc, ax=ax, label="Walk Score")
ax.set_xlabel("Interior size (sq ft, midpoint)")
ax.set_ylabel("Listed price (CAD)")
ax.set_title("Size vs price — coloured by walkability")
plt.tight_layout()
plt.show()

## Phase 5 — Market snapshot (pandas analytics)

In [ ]:
n = len(df)
snap = pd.Series({
    "listings": n,
    "pct_price_under_1p5m": round(100 * (df["listed_price_num"] < 1_500_000).mean(), 2),
    "pct_walk_ge_90": round(100 * (df["walk_score_num"] >= 90).mean(), 2),
    "mean_price_by_city": df.groupby("city")["listed_price_num"].mean().round(0).to_dict(),
    "cheapest_metro_by_mean_ppsf": (
        df.dropna(subset=["price_per_sqft", "city"])
        .groupby("city")["price_per_sqft"].mean().idxmin()
    ),
})
print(snap.to_string())

top_n = (
    df.dropna(subset=["price_per_sqft", "neighbourhood", "city"])
    .groupby(["city", "neighbourhood"], as_index=False)
    .agg(n=("price_per_sqft", "size"), mean_ppsf=("price_per_sqft", "mean"))
)
top_n = top_n[top_n["n"] >= 10].sort_values("mean_ppsf", ascending=False).head(8)
display(top_n)

## Phase 6 — Unsupervised: listing personas (K-Means)

We segment listings on **walk score**, **price per sqft**, and **size** — sensible axes for urban housing positioning.

In [ ]:
cluster_df = df[["walk_score_num", "price_per_sqft", "size_mid"]].dropna()
sample = cluster_df.sample(min(len(cluster_df), 5000), random_state=RANDOM_STATE)
Xs = SKStandardScaler().fit_transform(sample)
kmeans = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
sample = sample.copy()
sample["cluster"] = kmeans.fit_predict(Xs)
summ = (
    sample.groupby("cluster")
    .agg(
        n=("walk_score_num", "count"),
        walk=("walk_score_num", "mean"),
        ppsf=("price_per_sqft", "mean"),
        size=("size_mid", "mean"),
    )
    .round(2)
)
display(summ)

## Phase 7 — OLS: interpretable walkability premium

**Model:** log(price) ~ walk_score + log(size) + baths + property type FE + city FE.

Coefficients are **interpretable** for stakeholders; this complements ML in the next section.

In [ ]:
reg_df = df[
    ["listed_price_num", "walk_score_num", "size_mid", "bath_num", "type", "city"]
].dropna()
reg_df = reg_df[(reg_df["listed_price_num"] > 0) & (reg_df["size_mid"] > 0)].copy()
reg_df["log_price"] = np.log(reg_df["listed_price_num"])
reg_df["log_size"] = np.log(reg_df["size_mid"])
reg_df["type"] = reg_df["type"].astype("category")
reg_df["city"] = reg_df["city"].astype("category")

ols = smf.ols(
    "log_price ~ walk_score_num + log_size + bath_num + C(type) + C(city)",
    data=reg_df,
).fit()
print(ols.summary().tables[1])

coef_walk = ols.params.get("walk_score_num", np.nan)
if pd.notna(coef_walk):
    print("\nInterpretation (~log-linear):")
    print("  Approx % Δ price per +1 walk point:", round((np.exp(coef_walk) - 1) * 100, 3), "%")
    print("  Approx % Δ price per +10 walk points:", round((np.exp(coef_walk * 10) - 1) * 100, 2), "%")

## Phase 8 — Predictive models (portfolio-grade validation)

**Why GroupShuffleSplit by `city`?** A random split can leak **market-level structure** between train and test; grouping encourages **generalization across metros**.

**Target:** `log_price` = `log(listed_price_num)`.

**Metrics:** reported on **held-out group** — RMSE/MAE on log scale and **back-transformed** dollar RMSE for communication.

In [ ]:
modeling = df.dropna(
    subset=[
        "listed_price_num", "walk_score_num", "size_mid",
        "bath_num", "bed_num", "type", "city",
    ]
).copy()
modeling = modeling[
    (modeling["listed_price_num"] > 0) & (modeling["size_mid"] > 0)
]
modeling["log_price"] = np.log(modeling["listed_price_num"])

num_cols = ["walk_score_num", "size_mid", "bath_num", "bed_num"]
cat_cols = ["type", "city"]
X = modeling[num_cols + cat_cols]
y = modeling["log_price"]
groups = modeling["city"].astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

try:
    _ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    _ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", _ohe, cat_cols),
    ]
)


def rmse(y_a, y_b):
    return float(np.sqrt(mean_squared_error(y_a, y_b)))


def evaluate(name, estimator):
    pipe = Pipeline([("prep", preprocess), ("model", estimator)])
    pipe.fit(X_tr, y_tr)
    pred_log = pipe.predict(X_te)
    mae_log = mean_absolute_error(y_te, pred_log)
    r2 = r2_score(y_te, pred_log)
    y_true_usd = np.exp(y_te)
    y_pred_usd = np.clip(np.exp(pred_log), 1, None)
    return {
        "model": name,
        "RMSE_log": rmse(y_te, pred_log),
        "MAE_log": mae_log,
        "R2_log": r2,
        "RMSE_USD_approx": rmse(y_true_usd, y_pred_usd),
    }

rows = [
    evaluate("Ridge", Ridge(alpha=2.0, random_state=RANDOM_STATE)),
    evaluate(
        "RandomForest",
        RandomForestRegressor(
            n_estimators=300, max_depth=None, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1,
        ),
    ),
    evaluate(
        "GradientBoosting",
        GradientBoostingRegressor(random_state=RANDOM_STATE, max_depth=3, n_estimators=200),
    ),
]
metrics_df = pd.DataFrame(rows).round(4)
display(metrics_df)

print("\nTrain cities:", sorted(modeling.iloc[train_idx]["city"].unique()))
print("Test cities:", sorted(modeling.iloc[test_idx]["city"].unique()))

## Phase 9 — Feature importance by city (Random Forest)

In [ ]:
rf_results = []
for city_name, grp in df.dropna(subset=["listed_price_num"]).groupby("city"):
    sub = grp[
        ["listed_price_num", "walk_score_num", "size_mid", "bath_num", "bed_num"]
    ].copy().dropna()
    if len(sub) < 200:
        print(f"Skip {city_name}: only {len(sub)} complete rows")
        continue
    Xc = sub[["walk_score_num", "size_mid", "bath_num", "bed_num"]]
    yc = sub["listed_price_num"]
    rf = RandomForestRegressor(
        n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(Xc, yc)
    imps = pd.Series(rf.feature_importances_, index=Xc.columns).sort_values(ascending=False)
    rf_results.append(imps.reset_index().assign(city=city_name))

if rf_results:
    feat_imp = pd.concat(rf_results, ignore_index=True)
    feat_imp.columns = ["feature", "importance", "city"]
    display(feat_imp)
    pivot = feat_imp.pivot(index="feature", columns="city", values="importance").round(4)
    display(pivot)
else:
    print("Not enough per-city rows for Random Forest importance.")

## Phase 10 — Results recap (elevator pitch)

**Analytics**
- Market structure differs by **metro**; **size** and **walk score** move with **ask price** in the expected directions.

**Inference**
- OLS on **log-price** with **type and city fixed effects** gives a **stable, interpretable** walk score association (see Phase 7).

**Prediction**
- **Ridge / RF / Gradient Boosting** are compared with a **group-aware holdout** (Phase 8) so metrics are harder to “game” than a random split.

**Skills demonstrated**
- Pandas data engineering, visualization discipline, `statsmodels` + `sklearn` in one story, and **validation design**.

---

### Optional stretch (not run here)
Spatial cross-validation by neighbourhood, regularized high-cardinality encodings, or survival models if time-on-market appears — natural “Phase 11” talking points in interviews.